# Comparación de UCS, búsqueda voraz y A*

**Estudiante:** Johann Beskow 36923  
**Curso:** Inteligencia Artificial

In [1]:
ESTUDIANTE = 'Johann Beskow 36923'
CURSO = 'Inteligencia Artificial'

# Grafo dirigido: cada arista se representa como (destino, costo).
GRAFO = {
    'S': [('A', 2), ('B', 2)],
    'A': [('C', 2), ('D', 5)],
    'B': [('D', 2)],
    'C': [('G', 3)],
    'D': [('G', 6)],
    'G': []
}

# Estimación del costo restante hasta G.
HEURISTICA = {
    'S': 7,
    'A': 5,
    'B': 7,
    'C': 3,
    'D': 6,
    'G': 0
}

print(f'Estudiante: {ESTUDIANTE}')
print(f'Curso: {CURSO}')
print('Grafo:', GRAFO)
print('Heurística:', HEURISTICA)

Estudiante: Johann Beskow 36923
Curso: Inteligencia Artificial
Grafo: {'S': [('A', 2), ('B', 2)], 'A': [('C', 2), ('D', 5)], 'B': [('D', 2)], 'C': [('G', 3)], 'D': [('G', 6)], 'G': []}
Heurística: {'S': 7, 'A': 5, 'B': 7, 'C': 3, 'D': 6, 'G': 0}


In [2]:
def crear_nodo(estado, padre=None, accion=None, g=0, h=0, prioridad=0):
    return {
        'estado': estado,
        'padre': padre,
        'accion': accion,
        'g': g,
        'h': h,
        'f': g + h,
        'prioridad': prioridad
    }

def obtener_sucesores(estado):
    return GRAFO.get(estado, [])

def reconstruir_camino(nodo_final):
    camino = []
    actual = nodo_final
    while actual is not None:
        camino.append(actual['estado'])
        actual = actual['padre']
    return list(reversed(camino))

def formatear_frontera(frontera):
    return [
        f"{n['estado']}(g={n['g']},h={n['h']},f={n['f']},prio={n['prioridad']})"
        for n in frontera
    ]

def extraer_mejor(frontera):
    # El segundo criterio conserva el orden FIFO en caso de empate.
    indice, nodo = min(
        enumerate(frontera),
        key=lambda elemento: (elemento[1]['prioridad'], elemento[0])
    )
    frontera.pop(indice)
    return nodo

In [3]:
def busqueda_ucs(inicio, objetivo):
    raiz = crear_nodo(inicio, g=0, h=HEURISTICA[inicio], prioridad=0)
    frontera = [raiz]
    mejor_g = {inicio: 0}
    generados, expandidos, max_frontera, reaperturas = 1, 0, 1, 0
    estados_expandidos = set()

    print('\n--- TRAZA UCS (Prioridad: g) ---')
    while frontera:
        max_frontera = max(max_frontera, len(frontera))
        nodo = extraer_mejor(frontera)
        estado, g = nodo['estado'], nodo['g']
        if g > mejor_g[estado]:
            print(f"Obsoleto descartado: {estado} con g={g} (mejor_g actual={mejor_g[estado]})")
            continue
        print(f"Expandiendo: {estado} (g={g}, h={nodo['h']}, f={nodo['f']}, prio={nodo['prioridad']}) | Frontera: {formatear_frontera(frontera)}")
        if estado == objetivo:
            return {'camino': reconstruir_camino(nodo), 'costo': g, 'expandidos': expandidos, 'generados': generados, 'max_frontera': max_frontera, 'reaperturas': reaperturas}
        expandidos += 1
        estados_expandidos.add(estado)
        for sucesor, costo in obtener_sucesores(estado):
            nuevo_g = g + costo
            if sucesor not in mejor_g or nuevo_g < mejor_g[sucesor]:
                if sucesor in estados_expandidos:
                    reaperturas += 1
                mejor_g[sucesor] = nuevo_g
                frontera.append(crear_nodo(sucesor, nodo, f'{estado}->{sucesor}', nuevo_g, HEURISTICA[sucesor], nuevo_g))
                generados += 1
    return None

In [4]:
def busqueda_voraz(inicio, objetivo):
    h_inicio = HEURISTICA[inicio]
    raiz = crear_nodo(inicio, g=0, h=h_inicio, prioridad=h_inicio)
    frontera = [raiz]
    mejor_g = {inicio: 0}
    generados, expandidos, max_frontera, reaperturas = 1, 0, 1, 0
    estados_expandidos = set()

    print('\n--- TRAZA VORAZ (Prioridad: h) ---')
    while frontera:
        max_frontera = max(max_frontera, len(frontera))
        nodo = extraer_mejor(frontera)
        estado, g, h = nodo['estado'], nodo['g'], nodo['h']
        if g > mejor_g[estado]:
            print(f"Obsoleto descartado: {estado} con g={g} (mejor_g actual={mejor_g[estado]})")
            continue
        print(f"Expandiendo: {estado} (g={g}, h={h}, f={nodo['f']}, prio={nodo['prioridad']}) | Frontera: {formatear_frontera(frontera)}")
        if estado == objetivo:
            return {'camino': reconstruir_camino(nodo), 'costo': g, 'expandidos': expandidos, 'generados': generados, 'max_frontera': max_frontera, 'reaperturas': reaperturas}
        expandidos += 1
        estados_expandidos.add(estado)
        for sucesor, costo in obtener_sucesores(estado):
            nuevo_g = g + costo
            if sucesor not in mejor_g or nuevo_g < mejor_g[sucesor]:
                if sucesor in estados_expandidos:
                    reaperturas += 1
                mejor_g[sucesor] = nuevo_g
                frontera.append(crear_nodo(sucesor, nodo, f'{estado}->{sucesor}', nuevo_g, HEURISTICA[sucesor], HEURISTICA[sucesor]))
                generados += 1
    return None

In [5]:
def busqueda_a_estrella(inicio, objetivo):
    h_inicio = HEURISTICA[inicio]
    raiz = crear_nodo(inicio, g=0, h=h_inicio, prioridad=h_inicio)
    frontera = [raiz]
    mejor_g = {inicio: 0}
    generados, expandidos, max_frontera, reaperturas = 1, 0, 1, 0
    estados_expandidos = set()

    print('\n--- TRAZA A* (Prioridad: g + h) ---')
    while frontera:
        max_frontera = max(max_frontera, len(frontera))
        nodo = extraer_mejor(frontera)
        estado, g, f = nodo['estado'], nodo['g'], nodo['f']
        if g > mejor_g[estado]:
            print(f"Obsoleto descartado: {estado} con g={g} (mejor_g actual={mejor_g[estado]})")
            continue
        print(f"Expandiendo: {estado} (g={g}, h={nodo['h']}, f={f}, prio={nodo['prioridad']}) | Frontera: {formatear_frontera(frontera)}")
        if estado == objetivo:
            return {'camino': reconstruir_camino(nodo), 'costo': g, 'expandidos': expandidos, 'generados': generados, 'max_frontera': max_frontera, 'reaperturas': reaperturas}
        expandidos += 1
        estados_expandidos.add(estado)
        for sucesor, costo in obtener_sucesores(estado):
            nuevo_g = g + costo
            if sucesor not in mejor_g or nuevo_g < mejor_g[sucesor]:
                if sucesor in estados_expandidos:
                    reaperturas += 1
                mejor_g[sucesor] = nuevo_g
                h_sucesor = HEURISTICA[sucesor]
                frontera.append(crear_nodo(sucesor, nodo, f'{estado}->{sucesor}', nuevo_g, h_sucesor, nuevo_g + h_sucesor))
                generados += 1
    return None

In [6]:
res_ucs = busqueda_ucs('S', 'G')
res_voraz = busqueda_voraz('S', 'G')
res_a_star = busqueda_a_estrella('S', 'G')

resultados = {'UCS': res_ucs, 'Voraz': res_voraz, 'A*': res_a_star}
print('\n' + '=' * 85)
print(f"{'Resultado':<22} | {'UCS':<17} | {'Voraz':<17} | {'A*':<17}")
print('-' * 85)
print(f"{'Camino':<22} | {str(res_ucs['camino']):<17} | {str(res_voraz['camino']):<17} | {str(res_a_star['camino']):<17}")
print(f"{'Costo':<22} | {res_ucs['costo']:<17} | {res_voraz['costo']:<17} | {res_a_star['costo']:<17}")
print(f"{'Prioridad':<22} | {'g':<17} | {'h':<17} | {'g+h':<17}")
print(f"{'Expandidos (antes G)':<22} | {res_ucs['expandidos']:<17} | {res_voraz['expandidos']:<17} | {res_a_star['expandidos']:<17}")
print(f"{'Estados generados':<22} | {res_ucs['generados']:<17} | {res_voraz['generados']:<17} | {res_a_star['generados']:<17}")
print(f"{'Frontera máxima':<22} | {res_ucs['max_frontera']:<17} | {res_voraz['max_frontera']:<17} | {res_a_star['max_frontera']:<17}")
print(f"{'Reaperturas':<22} | {res_ucs['reaperturas']:<17} | {res_voraz['reaperturas']:<17} | {res_a_star['reaperturas']:<17}")
print('=' * 85)


--- TRAZA UCS (Prioridad: g) ---
Expandiendo: S (g=0, h=7, f=7, prio=0) | Frontera: []
Expandiendo: A (g=2, h=5, f=7, prio=2) | Frontera: ['B(g=2,h=7,f=9,prio=2)']
Expandiendo: B (g=2, h=7, f=9, prio=2) | Frontera: ['C(g=4,h=3,f=7,prio=4)', 'D(g=7,h=6,f=13,prio=7)']
Expandiendo: C (g=4, h=3, f=7, prio=4) | Frontera: ['D(g=7,h=6,f=13,prio=7)', 'D(g=4,h=6,f=10,prio=4)']
Expandiendo: D (g=4, h=6, f=10, prio=4) | Frontera: ['D(g=7,h=6,f=13,prio=7)', 'G(g=7,h=0,f=7,prio=7)']
Obsoleto descartado: D con g=7 (mejor_g actual=4)
Expandiendo: G (g=7, h=0, f=7, prio=7) | Frontera: []

--- TRAZA VORAZ (Prioridad: h) ---
Expandiendo: S (g=0, h=7, f=7, prio=7) | Frontera: []
Expandiendo: A (g=2, h=5, f=7, prio=5) | Frontera: ['B(g=2,h=7,f=9,prio=7)']
Expandiendo: C (g=4, h=3, f=7, prio=3) | Frontera: ['B(g=2,h=7,f=9,prio=7)', 'D(g=7,h=6,f=13,prio=6)']
Expandiendo: G (g=7, h=0, f=7, prio=0) | Frontera: ['B(g=2,h=7,f=9,prio=7)', 'D(g=7,h=6,f=13,prio=6)']

--- TRAZA A* (Prioridad: g + h) ---
Expandiend

## Tabla comparativa

| Resultado | UCS | Voraz | A* |
|---|---:|---:|---:|
| Camino | `S-A-C-G` | `S-A-C-G` | `S-A-C-G` |
| Costo | 7 | 7 | 7 |
| Prioridad | `g` | `h` | `g+h` |
| Expandidos antes de extraer G | 5 | 3 | 3 |
| Reaperturas | 0 | 0 | 0 |

## Respuestas de análisis

1. **¿Por qué Voraz y A* coinciden?** En este grafo, la heurística prioriza `A` sobre `B` (`5 < 7`) y `C` sobre `D` (`3 < 6`). Además, en el camino óptimo se mantiene `f = 7`. La heurística es admisible y consistente en este grafo. La admisibilidad por sí sola no garantiza que Voraz sea óptimo.

2. **¿Voraz garantiza el camino de menor costo?** No. Voraz prioriza solo `h(n)` y no considera el costo ya acumulado `g(n)`. Puede elegir un estado aparentemente cercano al objetivo que proviene de un camino muy caro.

3. **¿Qué ocurre con `h = 0` en A*?** La prioridad queda `f(n) = g(n) + 0 = g(n)`, por lo que A* coincide con UCS.

4. **¿Hubo reaperturas?** No hubo reaperturas en ninguno. En UCS, `D` fue descubierto con `g = 7` y luego mejorado a `g = 4` antes de ser expandido. Eso es una actualización de frontera, no una reapertura; la entrada vieja se descarta como obsoleta.

5. **¿Expandir menos estados significa encontrar un camino más barato?** No. Voraz expande 3 estados y obtiene costo 7, mientras UCS expande 5 y garantiza el menor costo. En otro grafo, Voraz podría expandir menos estados y devolver un camino más caro.